# 03 — Campaign Analysis
ROI, channel performance, A/B testing, objective comparison.

In [ ]:

import sys; sys.path.insert(0, '..')
from pipeline.load import get_connection
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

con = get_connection()
cp = con.execute("SELECT * FROM mart_campaign_performance").df()
events = con.execute("SELECT * FROM events").df()


## Campaign Revenue by Channel

In [ ]:

ch = cp.groupby('channel').agg(
    campaigns=('campaign_id','count'),
    total_revenue=('net_revenue','sum'),
    avg_conversion=('conversion_rate','mean'),
    avg_bounce=('bounce_rate','mean')
).round(4).sort_values('total_revenue', ascending=False)
print(ch)
fig, ax = plt.subplots(figsize=(9, 4))
ch['total_revenue'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Total Net Revenue by Channel'); ax.set_ylabel('Revenue ($)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()


## Conversion Rate vs Bounce Rate by Channel

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    cp['bounce_rate'], cp['conversion_rate'],
    c=cp['net_revenue'], cmap='RdYlGn', s=80, alpha=0.8
)
plt.colorbar(scatter, ax=ax, label='Net Revenue ($)')
for _, row in cp.iterrows():
    ax.annotate(row['channel'], (row['bounce_rate'], row['conversion_rate']),
                fontsize=7, alpha=0.7)
ax.set_xlabel('Bounce Rate'); ax.set_ylabel('Conversion Rate')
ax.set_title('Bounce Rate vs Conversion Rate (color = Revenue)')
plt.tight_layout(); plt.show()


## Campaign Objective Performance

In [ ]:

obj = cp.groupby('objective').agg(
    campaigns=('campaign_id','count'),
    avg_revenue=('net_revenue','mean'),
    avg_conversion=('conversion_rate','mean'),
    avg_expected_uplift=('expected_uplift','mean')
).round(4).sort_values('avg_revenue', ascending=False)
print(obj)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
obj['avg_revenue'].plot(kind='bar', ax=axes[0], color='mediumpurple', edgecolor='white')
axes[0].set_title('Avg Revenue by Objective'); plt.sca(axes[0]); plt.xticks(rotation=30, ha='right')
obj['avg_conversion'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Avg Conversion Rate by Objective'); plt.sca(axes[1]); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()


## A/B Test Analysis — Experiment Groups

In [ ]:

from scipy import stats

ab = events.groupby('experiment_group').agg(
    sessions=('session_id','nunique'),
    users=('customer_id','nunique'),
    purchases=('event_type', lambda x: (x=='purchase').sum()),
    bounces=('event_type', lambda x: (x=='bounce').sum()),
    avg_session_sec=('session_duration_sec','mean'),
).reset_index()
ab['conversion_rate'] = (ab['purchases'] / ab['sessions']).round(4)
ab['bounce_rate'] = (ab['bounces'] / ab['sessions']).round(4)
print(ab.to_string(index=False))

# Chi-squared test on Control vs Variant_A
control = ab[ab['experiment_group']=='Control'].iloc[0]
variant_a = ab[ab['experiment_group']=='Variant_A'].iloc[0]
ct = [[control['purchases'], control['sessions']-control['purchases']],
      [variant_a['purchases'], variant_a['sessions']-variant_a['purchases']]]
chi2, p, _, _ = stats.chi2_contingency(ct)
print(f"\nControl vs Variant_A — chi2={chi2:.3f}, p={p:.4f}")
print("Statistically significant" if p < 0.05 else "Not statistically significant")


## Expected vs Actual Uplift

In [ ]:

perf = cp[cp['net_revenue'].notna()].copy()
perf['actual_uplift_proxy'] = (perf['net_revenue'] / perf['net_revenue'].median() - 1).round(4)
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(perf))
ax.bar(x, perf['expected_uplift'], alpha=0.6, label='Expected Uplift', color='steelblue')
ax.bar(x, perf['actual_uplift_proxy'], alpha=0.6, label='Actual Uplift (proxy)', color='crimson')
ax.set_xticks(list(x)); ax.set_xticklabels(perf['campaign_id'].astype(str), rotation=90, fontsize=7)
ax.set_title('Expected vs Actual Uplift per Campaign')
ax.set_xlabel('Campaign ID'); ax.set_ylabel('Uplift')
ax.legend(); plt.tight_layout(); plt.show()
